In [1]:
%pylab inline 

import torch 
import torch.nn as nn
import torch.nn.functional as F 
import torchvision
from torchvision import transforms 
from tqdm import trange

Populating the interactive namespace from numpy and matplotlib


In [2]:
# Device Options 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load Data
norm_values = (0.5, 0.5, 0.5) 
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(norm_values,norm_values)
    ])
trainset = torchvision.datasets.CIFAR10(root="datasets/",download=False, train=True, transform=transform)
testset = torchvision.datasets.CIFAR10(root="datasets/", download=False, train=False, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, shuffle=True, batch_size=16)
testloader = torch.utils.data.DataLoader(testset, shuffle=False, batch_size=16)



In [3]:
# Write Training function to test on different models 
def train(model, epochs=5):
    model.train()
    # Loss And Optimization 
    loss_function = nn.CrossEntropyLoss()
    optim = torch.optim.Adam(params=model.parameters())

    # Training Loop 
    for _ in (t := trange(epochs)):
        for x,y in trainloader:
            x,y = x.to(device), y.to(device)

            # Forward Pass 
            outputs = model(x)
            loss = loss_function(outputs, y)

            # Update Weights
            optim.zero_grad()
            loss.backward()
            optim.step()

            # Training Accuracy 
            n_correct = torch.argmax(outputs, dim=1)
            acc = (n_correct == y).float().mean() * 100
        
        t.set_description(f"acc = {acc.item():.2f}%, loss = {loss.item():.2f}")
    return model
        


      


In [4]:
# Test Model Accuracy on test set
def show_model_accuracy(model):
    model.eval()
    with torch.no_grad():
        for x,y in testloader:
            x,y = x.to(device), y.to(device)
            outputs = model(x)
            n_correct = torch.argmax(outputs, dim=1)
            acc = (n_correct == y).float().mean()*100
        print(f"Val Accuracy = {acc.item():.2f}%")
     





In [5]:
# Model Evaluation 
def eval_model(model):
    trained_model = train(model)
    acc = show_model_accuracy(trained_model)
    return acc 


In [12]:
# Model with basic convnet architecture  
class BasicNet(nn.Module):
    def __init__(self):
        super(BasicNet, self).__init__()
        self.conv1 = nn.Conv2d(3, 12, 5) # 16, 3, 28, 28 
        self.pool = nn.MaxPool2d(2,2) # 16, 12, 14, 14 
        self.conv2 = nn.Conv2d(12, 24, 5) # 16, 24, 10, 10 
        self.fc1 = nn.Linear(24*10*10, 128) 
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)

    
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = x.view(-1, 24*10*10)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x 



In [7]:
# How good is our basic convnet?
bnet = BasicNet().to(device)
eval_model(bnet)

acc = 87.50%, loss = 0.60: 100%|██████████| 5/5 [01:04<00:00, 12.90s/it]
Val Accuracy = 62.50%


In [23]:
# AlexNet Model
# Paper: https://proceedings.neurips.cc/paper/2012/file/c399862d3b9d6b76c8436e924a68c45b-Paper.pdf
# Code: https://github.com/pytorch/vision/blob/master/torchvision/models/alexnet.py
alexnet = torchvision.models.AlexNet().to(device)
alexnet.classifier[1] = nn.Linear(9216,4096)
alexnet.classifier[4] = nn.Linear(4096,1024)
alexnet.classifier[6] = nn.Linear(1024,10)
alexnet
    

AlexNet(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(11, 11), stride=(4, 4), padding=(2, 2))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 192, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(192, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (avgpool): AdaptiveAvgPool2d(output_size=(6, 6))
  (classifier): Sequential(
    (0): Dropout(p=0.5, inplace=False)
    (1): Linear(in_features=9216, out_features=4096, bias=True)
 

In [20]:
# How Good is AlexNet?
eval_model(alexnet)

  0%|          | 0/5 [00:00<?, ?it/s]


RuntimeError: Given input size: (256x1x1). Calculated output size: (256x0x0). Output size is too small